# YOLO Object Detection

이미지 분류는 이미지 전체에 대해 하나의 정답을 예측하지만, 객체 탐지는 이미지 안에 어떤 객체가 있는지뿐 아니라 그 객체가 어디에 있는지도 함께 예측한다.

YOLO는 You Only Look Once의 약자이다. 이름 그대로 이미지를 여러 단계로 나누어 반복 처리하기보다, 한 번의 추론 과정에서 객체의 위치와 클래스를 빠르게 예측하는 계열의 객체 탐지 모델이다.

## Classification과 Object Detection의 차이

이미지 분류 모델의 출력은 보통 하나의 클래스이다.

예를 들어 SVHN 숫자 이미지를 입력하면 모델은 `0`부터 `9`까지의 클래스 중 하나를 예측한다.

반면 객체 탐지 모델은 다음 정보를 함께 출력한다.

- class: 탐지한 객체의 종류
- confidence: 모델이 해당 객체라고 판단한 확신도
- bounding box: 객체가 위치한 사각형 영역

따라서 객체 탐지는 단순한 분류보다 출력 구조가 복잡하다. 대신 실제 서비스에서는 훨씬 더 넓게 활용된다. 예를 들어 CCTV 사람 감지, 차량 탐지, 상품 진열 상태 확인, 불량품 검사, 웹캠 기반 실시간 인식 등이 객체 탐지 문제에 해당한다.

## Bounding Box, Confidence, IoU, NMS

객체 탐지에서 자주 등장하는 핵심 용어는 다음과 같다.

- Bounding Box: 객체를 둘러싸는 사각형 영역이다.
- Confidence Score: 모델이 해당 박스 안에 객체가 있다고 판단한 확신도이다.
- IoU(Intersection over Union): 두 박스가 얼마나 겹치는지 측정하는 값이다.
- NMS(Non-Maximum Suppression): 같은 객체에 대해 여러 박스가 중복 예측될 때, 가장 적절한 박스만 남기는 후처리 방식이다.

In [ ]:
# 필요한 라이브러리를 설치한다.
# - ultralytics: YOLO 모델 로드, 추론, 학습, 평가 기능 제공
# - opencv-python: 이미지/동영상 처리
%pip -q install ultralytics opencv-python

In [ ]:
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Video, display
from ultralytics import YOLO

## 사전학습 YOLO 모델 로드

https://docs.ultralytics.com/ko

https://docs.ultralytics.com/ko/tasks/detect

Ultralytics YOLO는 `YOLO("모델파일.pt")` 형태로 모델을 로드한다.

현재 공식 문서는 YOLO26 계열 예제를 기준으로 `yolo26n.pt`를 사용한다. 여기서 `n`은 nano 모델을 의미하며, 크기가 작아 실습용으로 적합하다.

단, 설치된 ultralytics 버전이나 실행 시점에 따라 제공되는 모델명이 다를 수 있다. 만약 `yolo26n.pt` 다운로드가 실패하면 같은 코드 구조에서 `yolo11n.pt` 또는 `yolov8n.pt`로 바꿔 실행하면 된다.

YOLO의 사전학습 모델은 보통 COCO 데이터셋을 기준으로 학습되어 있다. COCO는 사람, 자동차, 버스, 고양이, 강아지, 의자, 컵, 노트북 등 일상에서 자주 등장하는 객체 80개 클래스를 포함하는 대표적인 객체 탐지 데이터셋이다.

따라서 사전학습 YOLO 모델을 불러오면 별도의 추가 학습 없이도 COCO에 포함된 일반적인 객체를 탐지할 수 있다. `model.names`를 출력하면 현재 모델이 탐지할 수 있는 클래스 번호와 클래스 이름을 확인할 수 있다.